# 2. Finite geometry: rectangular prisms and tetrahedra

Point dipoles are the simplest source model. The primary application of
`cdfmm` is a mesh of **finite, uniformly magnetised bodies**: axis-aligned
rectangular prisms (micromagnetic cells) and tetrahedra (finite-element
meshes). Their near field is evaluated with the **exact analytical
interaction tensors** of the bodies, including each body's own demagnetising
field, while the far field uses multipole expansions of either the finite body
or its equivalent point dipole.

This tutorial covers the geometry records, the four model selectors, the
finite self field, the difference between point self *exclusion* and finite
self *inclusion*, and a comparison against the exact dense direct plan for
every source/target combination.

| Option | Values | Meaning |
|---|---|---|
| `source_geometry` | `POINT_DIPOLE`, `RECTANGULAR_PRISM`, `TETRAHEDRON` | physical shape of every source |
| `target_geometry` | `POINT`, `RECTANGULAR_PRISM`, `TETRAHEDRON` | physical shape of every target; a finite target receives the **volume-averaged** field |
| `source_sizes` / `target_sizes` | list of `RectangularPrism` | one common record, or one per body in user order |
| `source_tetrahedra` / `target_tetrahedra` | list of `Tetrahedron` | one common record, or one per body in user order |
| `near_field_source_model`, `near_field_target_model` | `EXACT_GEOMETRY` (default) or the point model | how list-1 (P2P) pairs treat the bodies |
| `far_field_source_model`, `far_field_target_model` | `EXACT_GEOMETRY` (default) or the point model | how P2M and L2P treat the bodies |

The four model selectors are independent, so the exact near field can be
combined with cheaper point far-field operators for controlled comparisons.

In [ ]:
import time

import numpy as np

import cdfmm
from tutorial_utils import (
    field_error_summary, lattice_positions, print_table, quiet_construction,
)

SG, TG = cdfmm.SourceGeometry, cdfmm.TargetGeometry
SM, TM = cdfmm.SourceModel, cdfmm.TargetModel

## Geometry records and the moment convention

`RectangularPrism(hx, hy, hz)` stores the **full** side lengths. A
`Tetrahedron` stores four vertices **relative to the body's representative
position**, which is normally its centroid, so one record can be shared by
every body of a regular mesh. Runtime input is always the total moment
$m = V\,M$ of each body; magnetisation is never inferred.

In [ ]:
side = 10.0e-9                                   # 10 nm cubes [m]
cube = cdfmm.RectangularPrism(side, side, side)

# A tetrahedron that fits inside the same cell, with its centroid at the origin.
vertices = side * np.array([[0.0, 0.0, 0.0], [0.8, 0.0, 0.0],
                            [0.0, 0.8, 0.0], [0.0, 0.0, 0.8]])
vertices -= vertices.mean(axis=0)
tetrahedron = cdfmm.Tetrahedron(vertices)

print(f"cube volume        {cube.volume:.3e} m^3")
print(f"tetrahedron volume {tetrahedron.volume:.3e} m^3 "
      f"(centroid offset {np.hypot(tetrahedron.centroid_offset.x, tetrahedron.centroid_offset.y):.1e})")

## The finite self field

A uniformly magnetised body produces a finite field inside itself. For a cube
the volume-averaged self field is exactly $H = -M/3$ (demagnetising factor
$1/3$ along every axis). This field is **physical** and is always kept for
finite bodies; only the *singular* self pair of a point dipole is removed, and
only through an explicit identity map. The dense direct plan evaluates the
exact tensors for every pair and is the reference used throughout.

In [ ]:
M = np.array([[6.0e5, -3.0e5, 1.5e5]])          # magnetisation [A/m]
origin = np.zeros((1, 3))

self_plan = cdfmm.DenseDirectPlan(
    origin, origin, SG.RECTANGULAR_PRISM, TG.POINT,
    source_sizes=[cube], static_precision="float64",
)
H_self = self_plan.evaluate(M * cube.volume)      # total moment m = V M
print("cube self field at its centre  ", H_self[0])
print("expected -M/3                  ", (-M / 3.0)[0])

# For a point dipole the same configuration is singular; the identity map
# removes it and the field is zero.
point_plan = cdfmm.DenseDirectPlan(
    origin, origin, SG.POINT_DIPOLE, TG.POINT,
    target_source_indices=[0], static_precision="float64",
)
print("point dipole with identity map ", point_plan.evaluate(M * cube.volume)[0])

## A lattice of prisms, with an exact dense reference

$6^3 = 216$ cubes of side 10 nm at 30 nm spacing carry a smooth magnetisation
pattern of amplitude $\sim 7\times10^5$ A/m. `DenseDirectPlan` with prism
sources **and** prism targets returns the exact volume-averaged field in every
receiving cube, including its self field; no identity map is passed because
finite self fields are physical.

In [ ]:
spacing = 3.0 * side
positions = lattice_positions(6, spacing)
phase = np.arange(len(positions), dtype=float)
magnetisation = np.column_stack((
    7.0e5 + 1.5e5 * np.sin(0.17 * phase),
    -3.0e5 + 2.0e5 * np.cos(0.11 * phase),
    4.0e5 * np.sin(0.07 * phase + 0.3),
))
moments = cube.volume * magnetisation

start = time.perf_counter()
dense = cdfmm.DenseDirectPlan(
    positions, positions, SG.RECTANGULAR_PRISM, TG.RECTANGULAR_PRISM,
    source_sizes=[cube], target_sizes=[cube], static_precision="float64",
)
H_dense = dense.evaluate(moments)
print(f"{len(positions)} prisms; dense construction {time.perf_counter() - start:.3f} s; "
      f"{dense.tensor_memory_bytes / 2**20:.2f} MiB of tensors")

## The same problem as an FMM plan

The plan options name the geometry once; every list-1 pair then uses the
exact prism-to-prism tensor and the far field uses the exact prism P2M and
volume-averaged L2P operators. The root box is derived from the bodies'
extents, not just their centres, so the whole lattice fits.

In [ ]:
def prism_options(order, depth, far_source=SM.EXACT_GEOMETRY,
                  far_target=TM.EXACT_GEOMETRY):
    options = cdfmm.UniformFmmOptions()
    options.precision = cdfmm.StaticPrecision.FLOAT64
    options.expansion_order = order
    options.tree.max_level = depth
    options.source_geometry = SG.RECTANGULAR_PRISM
    options.source_sizes = [cube]
    options.target_geometry = TG.RECTANGULAR_PRISM
    options.target_sizes = [cube]
    options.far_field_source_model = far_source
    options.far_field_target_model = far_target
    return options


start = time.perf_counter()
plan = cdfmm.UniformFmm(positions, positions, prism_options(order=6, depth=2))
setup = time.perf_counter() - start
H_fmm = plan.evaluate(moments)["H"]
print(f"\nFMM setup {setup:.3f} s; {plan.static_plan_statistics['p2p_interactions']} exact near-field pairs "
      f"of {len(positions) ** 2} total")
print("relative L2 error vs exact dense:", f"{field_error_summary(H_fmm, H_dense)['relative_l2']:.2e}")

## Point versus exact far-field models

The near field is always exact here; only the far-field treatment of the
bodies changes. For a centred **cube** the first shape correction to the point
dipole field appears at multipole degree five (the degree-three term is
proportional to the Laplacian and vanishes outside the source), so at low
orders the exact finite P2M/L2P operators cannot improve on the point
operators, and the four combinations give nearly the same error. Non-cubic
bodies, higher orders and closer packing change that picture; this is the
comparison to run for a specific mesh.

In [ ]:
rows = []
for order in (4, 6):
    for far_source, far_target, label in (
        (SM.POINT_DIPOLE, TM.POINT, "point P2M, point L2P"),
        (SM.EXACT_GEOMETRY, TM.POINT, "prism P2M, point L2P"),
        (SM.POINT_DIPOLE, TM.EXACT_GEOMETRY, "point P2M, prism L2P"),
        (SM.EXACT_GEOMETRY, TM.EXACT_GEOMETRY, "prism P2M, prism L2P"),
    ):
        with quiet_construction():
            candidate = cdfmm.UniformFmm(
                positions, positions, prism_options(order, 2, far_source, far_target))
        H = candidate.evaluate(moments)["H"]
        rows.append({"order": order, "far field": label,
                     "relative L2": field_error_summary(H, H_dense)["relative_l2"]})
print_table(rows, formats={"relative L2": ".3e"})

## Every source/target combination

All nine combinations of point, prism and tetrahedron sources and targets are
supported by the dense plan and by the FMM. The tetrahedron pairs and the
mixed prism/tetrahedron pairs use the analytical polyhedron surface
formulation (see the finite-geometry mathematics page). The check below runs
a smaller $5^3$ lattice through every pair, comparing the FMM with the exact
dense field. Point sources on a lattice coincide with the point targets, so
that pair needs the identity map; every finite pair keeps its self field.

The root box is set explicitly here. A regular lattice whose spacing divides
the automatic root box places every body **on** a box boundary, and a source
sitting at a box corner is the worst case for the multipole expansion (its
distance from the box centre is the half-diagonal, so the truncation error
decays slowly). Offsetting the lattice from the box boundaries restores the
normal convergence; tutorial 5 returns to this point.

In [ ]:
small_positions = lattice_positions(5, spacing, centre=(1.0e-8, 1.0e-8, 1.0e-8))
small_moments = cube.volume * magnetisation[: len(small_positions)]
identities = list(range(len(small_positions)))

geometries = {
    "point": dict(source=SG.POINT_DIPOLE, target=TG.POINT),
    "prism": dict(source=SG.RECTANGULAR_PRISM, target=TG.RECTANGULAR_PRISM),
    "tetra": dict(source=SG.TETRAHEDRON, target=TG.TETRAHEDRON),
}


def pair_options(source_name, target_name):
    options = cdfmm.UniformFmmOptions()
    options.precision = cdfmm.StaticPrecision.FLOAT64
    options.expansion_order = 6
    options.tree.max_level = 2
    options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
    options.tree.root_half_width = 3.0 * spacing
    options.source_geometry = geometries[source_name]["source"]
    options.target_geometry = geometries[target_name]["target"]
    if source_name == "prism":
        options.source_sizes = [cube]
    if target_name == "prism":
        options.target_sizes = [cube]
    if source_name == "tetra":
        options.source_tetrahedra = [tetrahedron]
    if target_name == "tetra":
        options.target_tetrahedra = [tetrahedron]
    if source_name == "point":
        options.fixed_target_source_indices = identities
    return options


rows = []
for source_name in geometries:
    for target_name in geometries:
        options = pair_options(source_name, target_name)
        reference = cdfmm.DenseDirectPlan(
            small_positions, small_positions,
            options.source_geometry, options.target_geometry,
            options.source_sizes, options.target_sizes,
            identities if source_name == "point" else [],
            "float64", options.source_tetrahedra, options.target_tetrahedra,
        ).evaluate(small_moments)
        with quiet_construction():
            candidate = cdfmm.UniformFmm(small_positions, small_positions, options)
        H = candidate.evaluate(
            small_moments,
            target_source_indices=identities if source_name == "point" else None,
        )["H"]
        rows.append({
            "source": source_name, "target": target_name,
            "near-field pairs": candidate.static_plan_statistics["p2p_interactions"],
            "relative L2": field_error_summary(H, reference)["relative_l2"],
        })
print_table(rows, formats={"relative L2": ".3e"})

## Self-interaction semantics in one table

| Source | Target coincides with source | Identity map supplied | Near-field self pair |
|---|---|---|---|
| point dipole | yes | yes | **omitted** (singular) |
| point dipole | yes | no | evaluated and singular: always supply the map |
| prism or tetrahedron | yes | yes or no | **kept**: the finite demagnetising self field |
| any | no | irrelevant | ordinary pair |

Identity is a matter of indices, so coincident coordinates without a map are
still two different particles.

## Optional: finite bodies on CUDA

The exact tensors are built on the host once and uploaded; evaluation streams
them on the device. The cell is skipped without a CUDA device.

In [ ]:
if cdfmm.cuda_full_available():
    options = prism_options(order=6, depth=2)
    options.backend = cdfmm.ExecutionBackend.CUDA_FULL
    with quiet_construction():
        cuda_plan = cdfmm.UniformFmm(positions, positions, options)
    H_cuda = cuda_plan.evaluate(moments)["H"]
    print("CUDA_FULL relative L2 error vs exact dense:",
          f"{field_error_summary(H_cuda, H_dense)['relative_l2']:.2e}",
          f"(packing {cuda_plan.p2p_execution_packing})")
else:
    print("CUDA is not available in this build.")

## Summary

- Prism and tetrahedron records are physical shapes; moments are total moments.
- The near field of every source/target pair is exact and keeps finite self fields.
- The four model selectors let the far field use the finite body or its point
  equivalent independently of the exact near field.
- `DenseDirectPlan` is the exact all-to-all reference for the same geometry.